# FERreal — Overnight Training

Trains SimpleCNN, ResNet-50, and ViT-Base/16 on FER-2013 in sequence.

**Before you start:**
1. Runtime → Change runtime type → GPU (T4 is fine, L4 is faster)
2. Make sure you have a Kaggle API token (kaggle.com → Settings → API → Create New Token → downloads `kaggle.json`)
3. Run cells 1–5 before bed to confirm setup works
4. Run cell 6 (the long one) and leave the laptop open, plugged in
5. In the morning, run cell 7 to generate figures

**Estimated time on T4:** ~5–7 hours total
**Estimated time on L4:** ~3–4 hours total

## Cell 1: Verify GPU is attached

In [ ]:
!nvidia-smi

If you don't see a GPU listed, go to Runtime → Change runtime type → GPU and re-run.

## Cell 2: Mount Google Drive

All checkpoints and results will be saved to Drive so they survive Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create a project folder in Drive
!mkdir -p /content/drive/MyDrive/FERreal/data
!mkdir -p /content/drive/MyDrive/FERreal/checkpoints
!mkdir -p /content/drive/MyDrive/FERreal/results

print('Drive mounted. Project folder ready at /content/drive/MyDrive/FERreal/')

## Cell 3: Clone the repo and install dependencies

In [ ]:
%cd /content
!rm -rf FERreal
!git clone https://github.com/aathithc/FERreal.git
%cd FERreal
!pip install -q -r requirements.txt
print('Repo cloned and dependencies installed.')

## Cell 4: Download FER-2013 from Kaggle

**You need a Kaggle API token.** Get one from kaggle.com → Settings → API → Create New Token. This downloads a `kaggle.json` file. Upload it when prompted below.

If you already have the dataset in Drive from a previous run, this cell will use the cached copy instead of re-downloading.

In [ ]:
import os
from pathlib import Path

DATA_DRIVE_PATH = Path('/content/drive/MyDrive/FERreal/data')
DATA_LOCAL_PATH = Path('/content/FERreal/data')

# Check if data already exists in Drive
if (DATA_DRIVE_PATH / 'fer2013' / 'fer2013.csv').exists():
    print('Dataset found in Drive — symlinking instead of re-downloading.')
    !rm -rf {DATA_LOCAL_PATH}
    !ln -s {DATA_DRIVE_PATH} {DATA_LOCAL_PATH}
else:
    print('Dataset not found in Drive. Downloading from Kaggle...')
    
    # Upload kaggle.json
    from google.colab import files
    print('Please upload your kaggle.json file:')
    uploaded = files.upload()
    
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    
    # Download to Drive directly so it persists
    !mkdir -p {DATA_DRIVE_PATH}/fer2013
    %cd {DATA_DRIVE_PATH}/fer2013
    !kaggle datasets download -d msambare/fer2013
    !unzip -q fer2013.zip
    !rm fer2013.zip
    
    %cd /content/FERreal
    !rm -rf {DATA_LOCAL_PATH}
    !ln -s {DATA_DRIVE_PATH} {DATA_LOCAL_PATH}

# Symlink checkpoints and results to Drive too
!rm -rf /content/FERreal/checkpoints /content/FERreal/results
!ln -s /content/drive/MyDrive/FERreal/checkpoints /content/FERreal/checkpoints
!ln -s /content/drive/MyDrive/FERreal/results /content/FERreal/results

print('\nData ready. Verifying:')
!ls -la /content/FERreal/data/

## Cell 5: Smoke test (2 epochs of SimpleCNN)

**Always run this before kicking off the overnight run.** It catches setup bugs in 5 minutes instead of 5 hours.

You should see:
- Training and validation loss numbers printing per epoch
- Loss decreasing from epoch 1 to epoch 2
- No errors

If anything fails here, fix it before going to bed.

In [ ]:
%cd /content/FERreal
!python -m src.training.train --config configs/default.yaml --model simple_cnn --epochs 2

## Cell 6: Overnight training run

This trains all three models in sequence. **Run this cell, then leave it alone.**

Before running:
1. Plug your laptop in
2. Set OS power settings to keep the laptop awake when the lid closes (or just leave the lid open)
3. Open the JavaScript console (F12 → Console) and paste the keep-alive script in the next cell's markdown to prevent idle disconnects

**Estimated time:**
- T4: ~5–7 hours
- L4: ~3–4 hours

Output goes to Drive at `/MyDrive/FERreal/checkpoints/{model}/best.pt` and `/MyDrive/FERreal/results/{model}/`.

**Anti-disconnect script** — paste this into your browser's JavaScript console (F12 → Console tab) before starting the long run:

```javascript
function ClickConnect(){console.log('Click'); document.querySelector('colab-connect-button').shadowRoot.querySelector('#connect').click()}
setInterval(ClickConnect, 60000)
```

In [ ]:
%cd /content/FERreal
import time

start = time.time()

# 1. SimpleCNN — 48x48 grayscale, ~30-45 min on T4
print('=' * 60)
print('Training SimpleCNN')
print('=' * 60)
!python -m src.training.train --config configs/default.yaml --model simple_cnn
print(f'\nSimpleCNN done at {(time.time()-start)/60:.1f} min\n')

# 2. ResNet-50 — 224x224 RGB, ~2 hours on T4
# Note: passing image_size and channels via CLI to override config defaults
print('=' * 60)
print('Training ResNet-50')
print('=' * 60)
!python -m src.training.train --config configs/default.yaml --model resnet --image-size 224 --channels 3
print(f'\nResNet-50 done at {(time.time()-start)/60:.1f} min\n')

# 3. ViT-Base/16 — 224x224 RGB, ~3 hours on T4
# If OOM, drop batch size to 32 by passing --batch-size 32
print('=' * 60)
print('Training ViT-Base/16')
print('=' * 60)
!python -m src.training.train --config configs/default.yaml --model vit --image-size 224 --channels 3 --batch-size 32
print(f'\nViT done at {(time.time()-start)/60:.1f} min\n')

print('=' * 60)
print(f'ALL TRAINING COMPLETE in {(time.time()-start)/60:.1f} min')
print('=' * 60)
!ls -la /content/drive/MyDrive/FERreal/checkpoints/
!ls -la /content/drive/MyDrive/FERreal/results/

## Cell 7: Generate figures (run in the morning)

After all three models are trained, this generates training curves, confusion matrices, per-class F1 plots, and the saliency grid.

In [ ]:
%cd /content/FERreal

# Run finalize_results.py for each model to write metrics.json from test split
!python scripts/finalize_results.py --model simple_cnn --checkpoint checkpoints/simple_cnn/best.pt
!python scripts/finalize_results.py --model resnet --checkpoint checkpoints/resnet/best.pt --image-size 224 --channels 3
!python scripts/finalize_results.py --model vit --checkpoint checkpoints/vit/best.pt --image-size 224 --channels 3

# Generate all the standard figures
!python scripts/generate_figures.py

# Generate Grad-CAM and attention rollout grid (this can take 5-10 min)
!python scripts/generate_saliency_grid.py

print('\nAll figures saved to /content/drive/MyDrive/FERreal/results/figures/')
!ls -la /content/drive/MyDrive/FERreal/results/figures/

## Cell 8: Download results to your local machine (optional)

Zips up the figures and metrics so you can drop them directly into the Overleaf report.

In [ ]:
import shutil
from google.colab import files

# Zip the figures and metrics together
shutil.make_archive(
    '/content/FERreal_results',
    'zip',
    '/content/drive/MyDrive/FERreal/results'
)

files.download('/content/FERreal_results.zip')
print('Download started. Check your Downloads folder.')